# 🎭 Aarya on Kaggle — ComfyUI identity pack
Kaggle variant of the Colab notebook. Differences that matter here:
- **Free GPU = 2× T4 (16 GB each), ~30 h/week quota, 12 h/session, idle timeout** (phone-verified
  account, no credit card). ComfyUI uses ONE GPU by default → **16 GB effective** → stills tier.
  (The 2nd T4 only helps via ComfyUI's experimental multigpu nodes — not turnkey for Wan-Animate-2.)
- **Persistence = datasets, not a drive:** upload once from your laptop as PRIVATE datasets:
  `aarya-pack` (zip of the repo's aarya/ folder, ~3 MB), `aarya-models-b` (stills stack ~3.5 GB),
  and for video GPUs elsewhere `aarya-models-a1` (diffusion_models+loras ~16.2 GB) +
  `aarya-models-a2` (text_encoders+clip_vision+vae ~7.7 GB). Private dataset cap = 20 GB each.
- `/kaggle/working` (20 GB, persists with Save Version) holds stills models + outputs.
- **Motion video (Wan-Animate-2 14B) does NOT fit a free Kaggle T4** — same verdict as Colab free.
  Use Runpod/vast.ai for video (see aarya/cloud/RUNPOD_SETUP.md).
Setup: Settings → Accelerator T4x2 · Internet ON · attach the datasets above.


In [ ]:
# 1) GPU check
import subprocess, os, torch
out = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader,nounits'],
                     capture_output=True, text=True).stdout.strip()
print(out)
vram = int(out.splitlines()[0].split(',')[-1].strip())
ngpu = torch.cuda.device_count()
TIER = 'A' if vram >= 20000 else 'B'
print(f"{ngpu} GPU(s), {vram} MiB each -> TIER {TIER} " +
      ("(video+stills)" if TIER=='A' else "(stills; video needs >=20GB single GPU: Runpod/vast.ai/Colab-Pro L4/A100)"))
if ngpu > 1: print("note: 2nd T4 idle by design; ComfyUI multigpu for Wan is experimental, not wired here")


In [ ]:
# 2) Storage plan + dataset detection
import glob, os
WORK='/kaggle/working'; os.makedirs(f'{WORK}/models', exist_ok=True); os.makedirs(f'{WORK}/aarya_output', exist_ok=True)
DS = glob.glob('/kaggle/input/aarya-models*')
PACK = glob.glob('/kaggle/input/aarya-pack*')
print('model datasets:', DS or 'NONE (stills models will download into /kaggle/working)')
print('pack dataset  :', PACK or 'NONE (drag workflow JSONs into the UI manually)')


In [ ]:
# 3) ComfyUI + deps + model dir wiring
import os, shutil
if not os.path.exists('/content/ComfyUI') and not os.path.exists('/kaggle/working/ComfyUI'):
    !git clone --depth 1 https://github.com/Comfy-Org/ComfyUI.git /kaggle/working/ComfyUI
os.chdir('/kaggle/working/ComfyUI')
!pip install -q -r requirements.txt
subs = ['checkpoints','loras','text_encoders','clip_vision','vae','inswapper','facerestore_models','diffusion_models']
srcs = {}
for d in DS:
    for sub in os.listdir(d):
        if sub in subs: srcs[sub] = os.path.join(d, sub)
for sub in subs:
    dst = f'/kaggle/working/ComfyUI/models/{sub}'
    if os.path.islink(dst): os.remove(dst)
    elif os.path.isdir(dst): shutil.rmtree(dst)
    if sub in srcs:
        os.symlink(srcs[sub], dst)          # from dataset (fast, persistent)
    else:
        os.makedirs(dst, exist_ok=True)     # will download into working (persists w/ Save Version)
print('wired:', {k: ('dataset' if k in srcs else 'working') for k in subs})


In [ ]:
# 4) Download missing stills-stack models into /kaggle/working (resumable)
import os
B = [
 ('checkpoints','v1-5-pruned-emaonly-fp16.safetensors',['https://huggingface.co/Comfy-Org/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly-fp16.safetensors','https://huggingface.co/sd-legacy/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly-fp16.safetensors']),
 ('loras','lcm-lora-sdv1-5.safetensors',['https://huggingface.co/latents/lcm-lora/resolve/main/lcm-lora-sdv1-5.safetensors']),
 ('inswapper','inswapper_128.onnx',['https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx']),
 ('facerestore_models','codeformer-v0.1.0.pth',['https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer-v0.1.0.pth']),
]
if TIER == 'A':
    print('TIER A: video models must come from your aarya-models-a1/a2 datasets (20GB private-dataset cap).')
    print('If they are missing, attach them or generate on Runpod/vast.ai instead.')
for sub, fn, urls in B:
    dst=f'/kaggle/working/ComfyUI/models/{sub}/{fn}'
    if os.path.exists(dst) and os.path.getsize(dst) > 10_000_000:
        print('skip', fn); continue
    ok=False
    for url in urls:
        print('get', fn, url)
        if os.system(f'wget -q -c -O "{dst}" "{url}"')==0 and os.path.getsize(dst)>10_000_000: ok=True; break
    if not ok: print('FAILED mirrors -> search filename on huggingface.co, wget to', dst)
print('model pass done')


In [ ]:
# 5) ReActor face-swap node
import os
if not os.path.exists('/kaggle/working/ComfyUI/custom_nodes/ComfyUI-ReActor'):
    !git clone --depth 1 https://github.com/Gourieff/ComfyUI-ReActor.git /kaggle/working/ComfyUI/custom_nodes/ComfyUI-ReActor
    !pip install -q -r /kaggle/working/ComfyUI/custom_nodes/ComfyUI-ReActor/requirements.txt
print('reactor ok')


In [ ]:
# 6) Workflows + Aarya refs
import shutil, glob, os
wfs = glob.glob('/kaggle/input/aarya-pack*/aarya/workflows/*.json') or glob.glob('/kaggle/input/aarya-pack*/workflows/*.json')
os.makedirs('/kaggle/working/aarya_workflows', exist_ok=True)
for f in wfs: shutil.copy(f, '/kaggle/working/aarya_workflows/')
refs = glob.glob('/kaggle/input/aarya-pack*/aarya/references/*') or glob.glob('/kaggle/input/aarya-pack*/references/*')
for f in refs: shutil.copy(f, '/kaggle/working/ComfyUI/input/')
shutil.copy('/kaggle/working/ComfyUI/input/aarya_ref_front_bust.png', '/kaggle/working/ComfyUI/input/aarya_ref.png') if os.path.exists('/kaggle/working/ComfyUI/input/aarya_ref_front_bust.png') else None
print('workflows:', os.listdir('/kaggle/working/aarya_workflows'))
print('inputs   :', os.listdir('/kaggle/working/ComfyUI/input'))
if not wfs: print('NO pack dataset -> drag the JSONs from your laptop into the ComfyUI tab instead')


In [ ]:
# 7) Launch ComfyUI + public tunnel (Internet toggle must be ON)
import subprocess, time, re, os
os.chdir('/kaggle/working/ComfyUI')
log=open('/kaggle/working/comfy.log','w')
subprocess.Popen(['python','main.py','--listen','0.0.0.0','--port','8188','--disable-auto-launch'],
                 stdout=log, stderr=subprocess.STDOUT)
if not os.path.exists('/kaggle/tmp/cloudflared'):
    os.makedirs('/kaggle/tmp', exist_ok=True)
    subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','-O','/kaggle/tmp/cloudflared'])
    os.chmod('/kaggle/tmp/cloudflared', 0o755)
tlog=open('/kaggle/working/tunnel.log','w')
subprocess.Popen(['/kaggle/tmp/cloudflared','tunnel','--no-autoupdate','--url','http://localhost:8188'],
                 stdout=tlog, stderr=subprocess.STDOUT)
url=None
for _ in range(60):
    time.sleep(2)
    m=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/kaggle/working/tunnel.log').read())
    if m: url=m.group(0); break
print('COMFYUI PUBLIC URL:', url or 'not up yet - check /kaggle/working/comfy.log')


In [ ]:
# 8) Outputs -> /kaggle/working/aarya_output (persists when you Save Version)
import shutil, os, glob
for f in glob.glob('/kaggle/working/ComfyUI/output/**/*', recursive=True):
    if os.path.isfile(f):
        dst=f.replace('/content/ComfyUI/output','/kaggle/working/aarya_output').replace('/kaggle/working/ComfyUI/output','/kaggle/working/aarya_output')
        os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy2(f, dst)
print('saved:', glob.glob('/kaggle/working/aarya_output/**/*', recursive=True)[-5:])
print('REMEMBER: File -> Save Version to keep working dir; STOP the session to save quota (idle burns hours)')


## Quota survival rules (Kaggle)
- ~30 GPU h/week rolling; **session wall-clock counts, idle included** → stop the session when done.
- 12 h hard session cap; idle timeout kills silent sessions → keep the tab alive while rendering.
- Phone verification required once for accelerators.
- P100 was sunset 15 Sep 2026; T4x2 is the free option and queues happen at peak.

## Load per tier
- TIER B (all free Kaggle sessions): `aarya_stills_facelock_cpu.json`, source face `aarya_face_direct.png`.
- TIER A (only if you ever get ≥20 GB single GPU): `aarya_video_wan_animate2_gpu.json`.
QA markers: black bindi between brows · silver stud LEFT nostril · centre-parted jet-black waves.
